<a href="https://colab.research.google.com/github/marinaalfarias/ia940/blob/main/Altura_e_In%C3%ADcio_de_notas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

#!pip install librosa==0.8.0
import librosa
import librosa.display

from IPython.display import Audio

# Abrir sample

In [ ]:
from google.colab import files
uploaded = files.upload()
nome_arquivo = next(iter(uploaded))
x, sr = librosa.load(nome_arquivo, mono=True)

print(f"Carregado com sucesso! Formato: {x.shape} | Taxa de amostragem: {sr} Hz")

In [ ]:
Audio(x, rate=sr)

In [ ]:
S = librosa.stft(x, n_fft=2048, hop_length=512, win_length=1024)
S = np.abs(S)
librosa.display.specshow(librosa.amplitude_to_db(S, ref=np.max), y_axis='log', x_axis='time', sr=sr)

# Visualização: trecho curto de audio

Representação no tempo. Note que estamos usando uma janela.

In [ ]:
t0 = 1 # Segundos
n0 = int(t0*sr)
frame_len = 1024 # Samples
t = np.linspace(0, frame_len/sr, frame_len)
x_ = x[n0:n0+frame_len]
x_ = x_ * np.hanning(len(x_))

plt.figure(figsize=(10,2))
plt.plot(t, x_)
#plt.plot(t, np.hanning(len(x_)))
plt.ylabel('Magnitude')
plt.xlabel('Tempo (s)')
plt.show()

Representação no espectro do trecho janelado acima.

In [ ]:
X_ = np.fft.fft(x_, len(x_)*2)

f = np.linspace(0, sr, len(X_))
plt.figure(figsize=(10,2))
plt.plot(f, 20*np.log10(np.abs(X_)))
plt.xlim([0,3000])
#plt.semilogy()
plt.show()

# Autocorrelação

###Teorema de Wiener-Khinchin

Esse teorema afirma que a auto-correlação de um sinal no domínio do tempo está diretamente relacionada à magnitude do espectro de potência do sinal no domínio da frequência. Mais precisamente, o teorema diz que:

$$
R_x(\tau) = \mathcal{F}^{-1} \left\{ |X(f)|^2 \right\}
$$

In [ ]:
X_ = np.fft.fft(x_, len(x_)*2)
pspect = X_ * np.conj(X_)
acc = np.fft.ifft(pspect)[0:len(x_)]

# Descartar parte imaginária (aparece por erro numérico)
#print(acc[0:10])
acc = np.real(acc)

# Remover partes menores que zero (irrelevantes!)
acc = np.maximum(acc, 0)


plt.figure(figsize=(10,2))
plt.plot(acc, label='???')
plt.ylabel('Magnitude')
plt.xlabel('???')
plt.legend()
plt.show()


## Encontrando primeiro pico na autocorrelação

In [ ]:
acc_up = np.zeros_like(acc) #Return an array of zeros with the same shape and type as a given array

#sobreamostragem
#cria uma versão de acc onde cada elemento original é repetido duas vezes no array acc_up.
for i in range(len(acc_up)):
  acc_up[i] = acc[int(i/2)]

plt.figure(figsize=(10,2))
plt.plot(acc, ':', label='???')
plt.plot(acc_up, label='???')
plt.ylabel('Magnitude')
plt.xlabel('???')
plt.legend()
plt.show()

In [ ]:
acc_sub = acc-acc_up

plt.figure(figsize=(10,2))
plt.plot(acc, ':', label='???')
plt.plot(acc_sub, label='???')
plt.ylabel('Magnitude')
plt.xlabel('???')
plt.legend()
plt.show()


In [ ]:
idx = np.argmax(acc_sub)

t0 = idx / sr
f0 = 1/t0

print("Frequencia fundamental: ", f0, "Hz")

plt.figure(figsize=(10,2))
plt.plot(acc, ':', label='???')
plt.plot(idx, acc[idx], 'o', label='???')
plt.ylabel('Magnitude')
plt.xlabel('???')
plt.legend()
plt.show()

## Refinamento do pico por interpolação polinomial

In [ ]:
y_ = acc[idx-3:idx+3]
plt.figure()
plt.plot(range(idx-3, idx+3), y_, '-o')
plt.plot(idx, acc[idx], 'o')
plt.show()

In [ ]:
# Fit polinomial
p = np.polyfit(range(idx-3, idx+3), y_, deg=2)
t_ = np.linspace(idx-4, idx+3, 100)
pt_ = np.polyval(p, t_)

plt.figure()
plt.plot(range(idx-3, idx+3), y_, '-o')
plt.plot(idx, acc[idx], 'o')
plt.plot(t_, pt_, ':')
plt.show()

In [ ]:
# Maximo do polinomio
dp = np.polyder(p)  #Returns the derivative of a polynomial
mx = np.roots(dp)[0]

plt.figure()
plt.plot(range(idx-3, idx+3), y_, '-o')
plt.plot(idx, acc[idx], 'o')
plt.plot(t_, pt_, ':')
plt.plot(mx, np.polyval(p, mx), 'o')
plt.show()

In [ ]:
t0 = mx / sr
f0 = 1/t0

print("Frequencia fundamental: ", f0, "Hz")

# Método Yin

In [ ]:
t0 = 1 # Posição inicial da janela, em segundos
n0 = int(t0*sr)
frame_len = 1024*2 # Samples
t = np.linspace(0, frame_len/sr, frame_len)
x_ = x[n0:n0+frame_len]

plt.figure(figsize=(10,2))
plt.plot(x_, label='Sinal janelado')
plt.ylabel('Amplitude')
plt.xlabel('Tempo (amostras)')
plt.legend()
plt.show()

## Função Diferença

In [ ]:
frame_d = int(frame_len/2)
fd = np.zeros(frame_d)
for i in range(frame_d):
  fd[i] = np.sum( ( x_[0:frame_d] - x_[i:frame_d+i] ) ** 2 )
  # Energia da diferença entre blocos com deslocamento relativo de i

plt.figure(figsize=(10,2))
plt.plot(fd, label='???')
plt.ylabel('Magnitude')
plt.xlabel('Deslocamento (amostras)')
plt.legend()
plt.show()

Reparem como a amplitude (eixo y) é pequena.

O método Yin sugere uma normalização.

## Funcao Diferença Normalizada Cumulativa Média

In [ ]:
idf = np.cumsum(fd)
plt.plot(idf, label='???')
cmndf = np.zeros_like(fd)

cmndf[0] = 1

for i in range(1, frame_d):
  cmndf[i] = fd[i] / ( (1/i)*idf[i])

plt.figure(figsize=(10,2))
plt.plot(cmndf, label='???')
plt.ylabel('Magnitude')
plt.xlabel('Deslocamento (amostras)')
plt.legend()
plt.show()

## Limiar e encontrar vales




In [ ]:
from scipy.signal import find_peaks

threshold = 0.1
c = np.minimum(cmndf, threshold) #minimum between cmndf and threshold
c *= -1                    #multiply all elements of c by -1
peaks, properties = find_peaks(c)
best_peak = min(peaks)

plt.figure(figsize=(10,2))
plt.plot(cmndf, label='????')
plt.plot(c)
plt.plot(best_peak, cmndf[best_peak], 'o', label='???')
plt.ylabel('Magnitude')
plt.xlabel('Deslocamento (amostras)')
plt.legend()
plt.show()

## Interpolação parabólica

In [ ]:
# Fit polinomial
cmndf_ = cmndf[best_peak-1:best_peak+2]
p = np.polyfit(range(best_peak-1,best_peak+2), cmndf_, deg=2)
t_ = np.linspace(best_peak-2, best_peak+2, 100)
pt_ = np.polyval(p, t_)
dp = np.polyder(p)
mx = np.roots(dp)[0]

plt.figure()
plt.plot(range(best_peak-1,best_peak+2), cmndf_, '-o', label='Função CMNDF')
plt.plot(best_peak, cmndf[best_peak], 'o', label='Vale achado na CMNDF')
plt.plot(t_, pt_, ':', label='Parábola que interpola a CMNDF')
plt.plot(mx, np.polyval(p, mx), 'o', label='Minimo da parábola')
plt.legend()
plt.xlabel('Deslocamento (amostras)')
plt.ylabel('Magnitude')
plt.show()

t0 = mx / sr
f0 = 1/t0

print("Frequencia fundamental: ", f0, "Hz")

# F0 ao longo do tempo

In [ ]:
# Framewise F0
f0 = librosa.yin(x, fmin=40, fmax=2000, sr=sr, frame_length=2048, win_length=None, hop_length=None, trough_threshold=0.1)
plt.figure(figsize=(10,2))
plt.plot(f0)
plt.ylabel('Frequência Fundamental (Hz)')
plt.xlabel('Tempo (quadros)')
plt.ylim([0,500])
plt.show()


Note que no começo não tem sinal, então F0 não é bem definido.

Depois de um período bastante estável, começam a aparecer oscilações quando a energia do sinal começa a cair.

In [ ]:
# F0 em escala MIDI
m0  = librosa.hz_to_midi(f0)
plt.figure(figsize=(10,2))
plt.plot(m0)
plt.show()

In [ ]:
# Energia ao longo do tempo
en = librosa.feature.rms(y=x, frame_length=2048).T
en = librosa.amplitude_to_db(en)
plt.figure(figsize=(10,2))
plt.plot(en)
plt.show()

# Detecção de Onsets

## Fluxo Espectral (spectral flux)

In [ ]:
X = np.abs(librosa.stft(x))

plt.figure()
plt.matshow(20*np.log10(X+1e-12))
plt.ylim([0, 300])
plt.show()

Xd = np.diff(X) #difference along the time axis
Xd_ = np.maximum(Xd, 0)

plt.figure()
plt.matshow(20*np.log10(Xd_+1e-12))
plt.ylim([0, 300])
plt.show()


flux = np.sum(Xd_, axis=0)  #sum along the frequency axis
#flux = librosa.amplitude_to_db(flux)

plt.figure(figsize=(12,2))
plt.plot(flux)
plt.autoscale(enable=True, axis='x', tight=True)
plt.show()
